<a href="https://colab.research.google.com/github/gauravjha201/Information_retrieval_practice/blob/main/IR_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import collections
import math

train_docs = [
    ("Taipei, Taiwan", "c"),
    ("Macao, Taiwan, Shanghai", "c"),
    ("Japan, Sapporo", "j"),
    ("Sapporo, Osaka, Taiwan", "j")
]

test_doc = "Taiwan, Taiwan, Sapporo"



In [ ]:
def train_naive_bayes(docs):
    # --- Step 1: Tokenization and Vocabulary Creation ---
    vocabulary = set()
    class_word_counts = collections.defaultdict(lambda: collections.defaultdict(int))
    class_total_words = collections.defaultdict(int)
    class_doc_counts = collections.defaultdict(int)

    for text, label in docs:
        class_doc_counts[label] += 1
        # Tokenize (split by comma/space and clean)
        words = [w.strip().lower() for w in text.replace(',', ' ').split()]
        for word in words:
            vocabulary.add(word)
            class_word_counts[label][word] += 1
            class_total_words[label] += 1

    # --- Step 2: Compute Prior Probabilities P(c) and P(j) ---
    total_docs = len(docs)
    priors = {label: count / total_docs for label, count in class_doc_counts.items()}

    return vocabulary, class_word_counts, class_total_words, priors

def classify(test_text, vocabulary, word_counts, total_words, priors):
    test_words = [w.strip().lower() for w in test_text.replace(',', ' ').split()]
    v_size = len(vocabulary)
    results = {}

    print(f"--- Classification for: '{test_text}' ---")

    for label in priors:
        # We use log probabilities to prevent numerical underflow
        log_prob = math.log(priors[label])

        print(f"\nClass {label}:")
        print(f"  Prior P({label}) = {priors[label]}")

        # --- Step 3 & 4: Compute Conditional Probabilities with Laplace Smoothing ---
        # Formula: P(word|class) = (count(word, class) + 1) / (total_words_in_class + V)
        for word in test_words:
            # Skip words not in the training vocabulary (optional, but standard)
            if word not in vocabulary:
                continue

            count = word_counts[label][word]
            # Conditional probability calculation
            cond_p = (count + 1) / (total_words[label] + v_size)
            log_prob += math.log(cond_p)
            print(f"  P('{word}'|{label}) = ({count} + 1) / ({total_words[label]} + {v_size}) = {cond_p:.4f}")

        results[label] = log_prob
        # Convert back from log for display (Posterior relative score)
        print(f"  Total Log Posterior for {label}: {log_prob:.4f}")

    # --- Step 5: Determine class based on Maximum Posterior Probability ---
    best_class = max(results, key=results.get)
    return best_class

# --- Execution ---
vocab, word_counts, total_words, priors = train_naive_bayes(train_docs)
prediction = classify(test_doc, vocab, word_counts, total_words, priors)

print(f"\nFinal Result: The test document is classified as Class '{prediction}'")

--- Classification for: 'Taiwan, Taiwan, Sapporo' ---

Class c:
  Prior P(c) = 0.5
  P('taiwan'|c) = (2 + 1) / (5 + 7) = 0.2500
  P('taiwan'|c) = (2 + 1) / (5 + 7) = 0.2500
  P('sapporo'|c) = (0 + 1) / (5 + 7) = 0.0833
  Total Log Posterior for c: -5.9506

Class j:
  Prior P(j) = 0.5
  P('taiwan'|j) = (1 + 1) / (5 + 7) = 0.1667
  P('taiwan'|j) = (1 + 1) / (5 + 7) = 0.1667
  P('sapporo'|j) = (2 + 1) / (5 + 7) = 0.2500
  Total Log Posterior for j: -5.6630

Final Result: The test document is classified as Class 'j'


In [ ]:
import math

docs = [
    ("Taipei Taiwan", "C"),
    ("Macao Taiwan Shanghai", "C"),
    ("Japan Sapporo", "J"),
    ("Sapporo Osaka Taiwan", "J")
]

test_doc = ["Taiwan", "Taiwan", "Sapporo"]

vocab = set()
class_counts = {"C": 0, "J": 0}

word_counts_C = {}
word_counts_J = {}

total_words_C = 0
total_words_J = 0

# Step 2: Count words
for text, label in docs:
    words = text.split()
    class_counts[label] += 1

    for word in words:
        vocab.add(word)

        if label == "C":
            word_counts_C[word] = word_counts_C.get(word, 0) + 1
            total_words_C += 1
        else:
            word_counts_J[word] = word_counts_J.get(word, 0) + 1
            total_words_J += 1

vocab_size = len(vocab)
total_docs = len(docs)

# Step 3: Priors
P_C = class_counts["C"] / total_docs
P_J = class_counts["J"] / total_docs

# Step 4: Start log probabilities
log_C = math.log(P_C)
log_J = math.log(P_J)

# Step 5: Calculate likelihoods
for word in test_doc:

    # For Class C
    count_C = word_counts_C.get(word, 0)
    prob_C = (count_C + 1) / (total_words_C + vocab_size)
    log_C += math.log(prob_C)

    # For Class J
    count_J = word_counts_J.get(word, 0)
    prob_J = (count_J + 1) / (total_words_J + vocab_size)
    log_J += math.log(prob_J)

# Step 6: Compare
print("Log Probability C:", log_C)
print("Log Probability J:", log_J)

if log_C > log_J:
    print("Predicted Class: C")
else:
    print("Predicted Class: J")

Log Probability C: -5.950642552587727
Log Probability J: -5.662960480135946
Predicted Class: J
